[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/27.%20The%20Integrated%20Berth%20%26%20Crane%20Allocation%20Problem%20%28BAP-QCAP%29/P27-Tier-5_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 27. The Integrated Berth & Crane Allocation Problem (BAP-QCAP)

## Tier 5: Integrated Digital Twin (System-of-Systems Simulation)

### Goal

This notebook builds a small, reproducible simulation scaffold for a container terminal. It represents vessels, quay cranes, berth segments, weather, and IoT sensor readings, then advances their simulated state over time and reports selected operational indicators.

The implementation illustrates how terminal subsystems and time-stamped information can be represented in one software system. It does **not** implement a berth-and-crane allocation optimizer, a predictive model, a rolling-horizon procedure, or a live connection to terminal equipment. Those capabilities are described as possible components of a production digital twin, but they are not computed in this notebook. In particular, vessels are not assigned to berth segments and cranes are not assigned to vessels; the berth and crane utilization indicators therefore reflect the assignments present in the simulation state, not an optimized operating plan.

### Inputs and outputs

The inputs are the explicitly defined vessel records, terminal dimensions, initial weather, and simulation settings in the code. Random weather changes and crane failures use NumPy's seeded random-number generator. The outputs are an event log, time histories of berth-segment and crane utilization, and visual summaries of the simulated state. No measured terminal data or externally sourced performance results are used.

### Model assumptions and scope

- Time is measured in hours, quay length and vessel length in meters, draft and water depth in meters, and workload in TEU.
- The simulation advances in fixed time steps. A vessel changes operational status when its ETA or any service-time fields satisfy the relevant transition condition.
- Weather changes and crane failures are generated stochastically for demonstration. They are not calibrated to a real terminal or a statistical failure model.
- Berth segments and cranes are represented as separate resources. The code does not implement spatial berth allocation, crane allocation, vessel service scheduling, yard operations, or vessel handling-time calculations.
- The sensor network is an in-memory representation updated from the simulated state. It is not a real-time data integration system.
- Prediction, optimization, and scenario-analysis result containers are placeholders; no predictive or optimization calculations are performed.

### Simulation logic

Let $S_t$ denote the collection of simulated terminal records at time $t$, including vessel statuses, crane statuses, berth-segment occupancy, weather history, and sensor data. A simulation step advances time by $\Delta t$ and applies the state-update procedures in sequence:

$$
S_{t+\Delta t} = F_{\Delta t}(S_t),
$$

where $F_{\Delta t}$ is the combined set of update operations implemented by `DigitalTwinSystem.update_time`. This is a discrete-time simulation transition, not an optimization recurrence: the notebook does not choose decisions by minimizing an objective function or solving allocation constraints.

The simulation records berth utilization as the fraction of berth segments occupied at the current time, and crane utilization as the fraction of cranes whose status is `ASSIGNED`:

$$
U^{\mathrm{berth}}_t = \frac{\text{occupied berth segments at }t}{\text{number of berth segments}},
\qquad
U^{\mathrm{crane}}_t = \frac{\text{assigned cranes at }t}{\text{number of cranes}}.
$$

These are state indicators, not measures of an optimal plan. Since this implementation has no assignment procedure, they remain zero unless the simulation state is assigned by some other explicit code.

### What to look for

- How the data classes represent the state of terminal assets.
- How a time step updates weather, sensor readings, vessel statuses, crane statuses, and KPI histories.
- How stochastic events are recorded and how the visualizations summarize the resulting state.
- Which capabilities a fuller digital twin would need beyond this simulation scaffold.

In [ ]:
# Import the libraries used by the simulation, data classes, and visualizations.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum

# Fixed seeds make the stochastic demonstration reproducible.
np.random.seed(42)

plt.style.use("seaborn-v0_8")

print("Libraries imported successfully.")

In [ ]:
# Enumerations make the permitted operational states explicit.
class VesselStatus(Enum):
    """Operational states used for vessels in the simulation."""
    APPROACHING = "approaching"
    WAITING = "waiting"
    BERTHING = "berthing"
    SERVICING = "servicing"
    DEPARTING = "departing"
    COMPLETED = "completed"


class CraneStatus(Enum):
    """Operational states used for quay cranes in the simulation."""
    AVAILABLE = "available"
    ASSIGNED = "assigned"
    MAINTENANCE = "maintenance"
    BREAKDOWN = "breakdown"
    IDLE = "idle"


class WeatherCondition(Enum):
    """Weather categories used by the productivity-factor calculation."""
    CLEAR = "clear"
    CLOUDY = "cloudy"
    RAIN = "rain"
    STORM = "storm"
    FOG = "fog"


@dataclass
class Vessel:
    """Simulated vessel record; times are hours from the simulation start."""
    id: int
    name: str
    length: float  # meters
    draft: float  # meters
    workload: float  # TEU
    arrival_time: float  # hours from start
    eta: float  # estimated time of arrival, in hours from start
    priority: float = 1.0
    status: VesselStatus = VesselStatus.APPROACHING
    assigned_berth: Optional[float] = None
    assigned_cranes: int = 0
    service_start_time: Optional[float] = None
    service_end_time: Optional[float] = None
    cost_per_hour: float = 1000.0
    max_cranes: int = 5

    def update_eta(self, new_eta: float) -> None:
        """Update both ETA fields so the vessel record remains consistent."""
        self.eta = new_eta
        self.arrival_time = new_eta


@dataclass
class Crane:
    """Simulated quay-crane record."""
    id: int
    position: float  # meters along the quay
    productivity: float  # TEU per hour
    status: CraneStatus = CraneStatus.AVAILABLE
    assigned_vessel: Optional[int] = None
    maintenance_until: Optional[float] = None
    breakdown_until: Optional[float] = None
    utilization_history: List[float] = field(default_factory=list)

    def assign_to_vessel(self, vessel_id: int) -> None:
        """Record an assignment and update the crane's operational state."""
        self.assigned_vessel = vessel_id
        self.status = CraneStatus.ASSIGNED

    def release(self) -> None:
        """Clear an assignment and make the crane available."""
        self.assigned_vessel = None
        self.status = CraneStatus.AVAILABLE


@dataclass
class BerthSegment:
    """A quay interval and its simulated occupancy information."""
    start_position: float
    end_position: float
    length: float
    occupied_by: Optional[int] = None
    occupied_until: Optional[float] = None
    depth: float = 16.0  # meters

    def is_available(self, current_time: float) -> bool:
        """Return whether this segment is unoccupied at the specified time."""
        return (
            self.occupied_by is None
            or self.occupied_until is None
            or self.occupied_until <= current_time
        )


@dataclass
class WeatherData:
    """Weather observation with a simple illustrative productivity factor."""
    timestamp: float
    condition: WeatherCondition
    wind_speed: float  # km/h
    visibility: float  # km
    temperature: float  # Celsius
    wave_height: float  # meters

    def get_productivity_factor(self) -> float:
        """Return a bounded factor based on category, wind, and visibility."""
        factors = {
            WeatherCondition.CLEAR: 1.0,
            WeatherCondition.CLOUDY: 0.95,
            WeatherCondition.RAIN: 0.85,
            WeatherCondition.STORM: 0.6,
            WeatherCondition.FOG: 0.7,
        }
        factor = factors[self.condition]

        # Apply the stronger wind reduction first so the two thresholds are distinct.
        if self.wind_speed > 50:
            factor *= 0.6
        elif self.wind_speed > 30:
            factor *= 0.8

        if self.visibility < 1.0:
            factor *= 0.7

        return max(0.3, factor)


@dataclass
class IoTSensor:
    """In-memory representation of a sensor and its most recent data."""
    id: str
    type: str
    location: str
    last_update: float
    data: Dict = field(default_factory=dict)
    status: str = "active"

    def update_data(self, new_data: Dict, timestamp: float) -> None:
        """Merge a reading into the sensor record and timestamp the update."""
        self.data.update(new_data)
        self.last_update = timestamp


print("Simulation data structures defined successfully.")

In [ ]:
# The system coordinates simulated components and their state updates.
class DigitalTwinSystem:
    """Discrete-time terminal simulation for the BAP-QCAP context."""

    def __init__(
        self,
        quay_length: float = 1200,
        num_cranes: int = 15,
        num_berth_segments: int = 30,
        planning_horizon: float = 48,
    ):
        if quay_length <= 0 or num_cranes <= 0 or num_berth_segments <= 0:
            raise ValueError("Quay length and resource counts must be positive.")
        if planning_horizon <= 0:
            raise ValueError("Planning horizon must be positive.")

        self.quay_length = quay_length
        self.num_cranes = num_cranes
        self.num_berth_segments = num_berth_segments
        self.planning_horizon = planning_horizon

        self.vessels: Dict[int, Vessel] = {}
        self.cranes: Dict[int, Crane] = {}
        self.berth_segments: List[BerthSegment] = []
        self.weather_history: List[WeatherData] = []
        self.iot_sensors: Dict[str, IoTSensor] = {}

        self.current_time = 0.0
        self.simulation_speed = 1.0

        self.kpi_history: Dict[str, List[float]] = {
            "berth_utilization": [],
            "crane_utilization": [],
            "vessel_turnaround": [],
            "schedule_adherence": [],
            "productivity": [],
            "disruption_count": [],
            "prediction_accuracy": [],
        }

        # These containers reserve places for results; this notebook does not
        # implement optimization, scenario analysis, or predictive models.
        self.optimization_results: List[Dict] = []
        self.scenario_analysis_results: List[Dict] = []
        self.arrival_predictions: Dict[int, float] = {}
        self.equipment_failure_predictions: Dict[int, float] = {}

        self.initialize_system()

    def initialize_system(self) -> None:
        """Create quay segments, cranes, sensors, and the initial weather record."""
        print("Initializing terminal simulation...")

        segment_length = self.quay_length / self.num_berth_segments
        for i in range(self.num_berth_segments):
            self.berth_segments.append(
                BerthSegment(
                    start_position=i * segment_length,
                    end_position=(i + 1) * segment_length,
                    length=segment_length,
                )
            )

        for i in range(self.num_cranes):
            crane_id = i + 1
            self.cranes[crane_id] = Crane(
                id=crane_id,
                position=i * (self.quay_length / self.num_cranes),
                productivity=30.0,
            )

        self.initialize_iot_sensors()
        self.initialize_weather()

        print(f"Berth segments: {self.num_berth_segments} ({self.quay_length} m total)")
        print(f"Quay cranes: {self.num_cranes}")
        print(f"In-memory sensors: {len(self.iot_sensors)}")
        print(f"Planning horizon: {self.planning_horizon} hours")

    def initialize_iot_sensors(self) -> None:
        """Create sensor records associated with berth, crane, and environment."""
        for i in range(self.num_berth_segments):
            sensor = IoTSensor(
                id=f"berth_{i}",
                type="berth_occupancy",
                location=f"Segment {i}",
                last_update=0.0,
                data={"occupied": False, "vessel_id": None},
            )
            self.iot_sensors[sensor.id] = sensor

        for crane_id in range(1, self.num_cranes + 1):
            sensor = IoTSensor(
                id=f"crane_{crane_id}",
                type="crane_status",
                location=f"Crane {crane_id}",
                last_update=0.0,
                data={"status": "available", "utilization": 0.0},
            )
            self.iot_sensors[sensor.id] = sensor

        sensor = IoTSensor(
            id="env_1",
            type="environmental",
            location="Terminal entrance",
            last_update=0.0,
            data={"temperature": 20.0, "humidity": 65.0, "wind_speed": 10.0},
        )
        self.iot_sensors[sensor.id] = sensor

    def initialize_weather(self) -> None:
        """Set the initial weather record used by subsequent simulation steps."""
        self.weather_history.append(
            WeatherData(
                timestamp=0.0,
                condition=WeatherCondition.CLEAR,
                wind_speed=10.0,
                visibility=10.0,
                temperature=20.0,
                wave_height=0.5,
            )
        )

    def add_vessel(self, vessel: Vessel) -> None:
        """Add a vessel, requiring its identifier to be unique."""
        if vessel.id in self.vessels:
            raise ValueError(f"Vessel ID {vessel.id} is already in the system.")
        self.vessels[vessel.id] = vessel
        print(f"Vessel {vessel.name} (ID: {vessel.id}) added to the simulation.")

    def update_time(self, delta_time: float) -> None:
        """Advance the clock and update each simulated subsystem once."""
        if delta_time <= 0:
            raise ValueError("The time step must be positive.")
        self.current_time += delta_time
        self.update_weather()
        self.update_iot_sensors()
        self.update_vessel_statuses()
        self.update_crane_statuses()
        self.calculate_kpis()

    def update_weather(self) -> None:
        """Occasionally append a randomly generated weather observation."""
        if np.random.random() < 0.1:
            condition = np.random.choice(list(WeatherCondition))
            self.weather_history.append(
                WeatherData(
                    timestamp=self.current_time,
                    condition=condition,
                    wind_speed=float(np.random.uniform(5, 60)),
                    visibility=float(np.random.uniform(0.5, 15)),
                    temperature=float(np.random.uniform(10, 35)),
                    wave_height=float(np.random.uniform(0.1, 3.0)),
                )
            )

    def update_iot_sensors(self) -> None:
        """Refresh sensor records from the current simulated asset states."""
        for i, segment in enumerate(self.berth_segments):
            sensor = self.iot_sensors.get(f"berth_{i}")
            if sensor is not None:
                occupied = not segment.is_available(self.current_time)
                sensor.update_data(
                    {"occupied": occupied, "vessel_id": segment.occupied_by},
                    self.current_time,
                )

        for crane_id, crane in self.cranes.items():
            sensor = self.iot_sensors.get(f"crane_{crane_id}")
            if sensor is not None:
                utilization = float(crane.status == CraneStatus.ASSIGNED)
                sensor.update_data(
                    {"status": crane.status.value, "utilization": utilization},
                    self.current_time,
                )

    def update_vessel_statuses(self) -> None:
        """Apply status transitions when ETA or service-time conditions are met."""
        for vessel in self.vessels.values():
            if vessel.status == VesselStatus.APPROACHING and vessel.eta <= self.current_time:
                vessel.status = VesselStatus.WAITING
            elif vessel.status == VesselStatus.WAITING and vessel.assigned_berth is not None:
                vessel.status = VesselStatus.BERTHING
            elif vessel.status == VesselStatus.BERTHING and vessel.service_start_time is not None:
                if vessel.service_start_time <= self.current_time:
                    vessel.status = VesselStatus.SERVICING
            elif vessel.status == VesselStatus.SERVICING and vessel.service_end_time is not None:
                if vessel.service_end_time <= self.current_time:
                    vessel.status = VesselStatus.DEPARTING
            elif vessel.status == VesselStatus.DEPARTING:
                vessel.status = VesselStatus.COMPLETED

    def update_crane_statuses(self) -> None:
        """Recover unavailable cranes and generate illustrative random failures."""
        for crane in self.cranes.values():
            if (
                crane.status == CraneStatus.MAINTENANCE
                and crane.maintenance_until is not None
                and crane.maintenance_until <= self.current_time
            ):
                crane.status = CraneStatus.AVAILABLE
                crane.maintenance_until = None

            if (
                crane.status == CraneStatus.BREAKDOWN
                and crane.breakdown_until is not None
                and crane.breakdown_until <= self.current_time
            ):
                crane.status = CraneStatus.AVAILABLE
                crane.breakdown_until = None

            if crane.status == CraneStatus.AVAILABLE and np.random.random() < 0.001:
                repair_time = float(np.random.uniform(2, 8))
                crane.status = CraneStatus.BREAKDOWN
                crane.breakdown_until = self.current_time + repair_time
                print(
                    f"Crane {crane.id} breakdown; estimated repair completion "
                    f"at {crane.breakdown_until:.1f} hours."
                )

    def calculate_kpis(self) -> None:
        """Append berth-segment and crane utilization for the current time."""
        occupied_segments = sum(
            not segment.is_available(self.current_time)
            for segment in self.berth_segments
        )
        self.kpi_history["berth_utilization"].append(
            occupied_segments / self.num_berth_segments
        )

        assigned_cranes = sum(
            crane.status == CraneStatus.ASSIGNED for crane in self.cranes.values()
        )
        self.kpi_history["crane_utilization"].append(
            assigned_cranes / self.num_cranes
        )


print("Terminal simulation system class defined successfully.")

In [ ]:
# Instantiate a terminal with the stated dimensions and resource counts.
digital_twin = DigitalTwinSystem(
    quay_length=1200,
    num_cranes=15,
    num_berth_segments=30,
    planning_horizon=48,
)

print("\n=== Simulation initialized ===")
print(f"Simulation time: {digital_twin.current_time:.1f} hours")
print(f"Berth segments: {len(digital_twin.berth_segments)}")
print(f"Cranes: {len(digital_twin.cranes)}")
print(f"In-memory sensors: {len(digital_twin.iot_sensors)}")
print(f"Weather records: {len(digital_twin.weather_history)}")

In [ ]:
# The sample vessels provide records whose ETA transitions can be observed.
vessels_to_add = [
    Vessel(
        id=1, name="Maersk Evergiven", length=399, draft=15.5,
        workload=1200, arrival_time=2.5, eta=2.5, priority=1.2, max_cranes=5,
    ),
    Vessel(
        id=2, name="COSCO Shanghai", length=366, draft=14.2,
        workload=950, arrival_time=4.0, eta=4.0, priority=1.0, max_cranes=4,
    ),
    Vessel(
        id=3, name="Hapag Lloyd Atlas", length=350, draft=13.8,
        workload=800, arrival_time=6.5, eta=6.5, priority=0.8, max_cranes=4,
    ),
    Vessel(
        id=4, name="MSC Oscar", length=400, draft=16.0,
        workload=1500, arrival_time=8.0, eta=8.0, priority=1.5, max_cranes=5,
    ),
    Vessel(
        id=5, name="ONE Stork", length=340, draft=13.5,
        workload=750, arrival_time=10.0, eta=10.0, priority=0.9, max_cranes=3,
    ),
]

for vessel in vessels_to_add:
    digital_twin.add_vessel(vessel)

print("\n=== Vessel records ===")
for vessel in digital_twin.vessels.values():
    print(
        f"{vessel.name}: ETA {vessel.eta:.1f} hours, "
        f"priority {vessel.priority:.1f}, length {vessel.length} m"
    )

In [ ]:
def simulate_digital_twin_operation(
    digital_twin: DigitalTwinSystem,
    duration_hours: float = 24,
) -> List[Dict]:
    """Advance the simulation in half-hour steps and record detected events."""
    if duration_hours < 0:
        raise ValueError("Simulation duration cannot be negative.")

    print(f"\n=== Simulating {duration_hours:g} hours ===")
    time_step = 0.5
    num_steps = int(duration_hours / time_step)
    operation_log: List[Dict] = []

    for step in range(num_steps):
        previous_weather = digital_twin.weather_history[-1]
        previous_vessel_statuses = {
            vessel_id: vessel.status
            for vessel_id, vessel in digital_twin.vessels.items()
        }
        previous_crane_statuses = {
            crane_id: crane.status
            for crane_id, crane in digital_twin.cranes.items()
        }

        digital_twin.update_time(time_step)
        current_weather = digital_twin.weather_history[-1]

        # Compare states across the step to record transitions that occurred
        # inside update_time, including arrivals and newly generated failures.
        for vessel_id, vessel in digital_twin.vessels.items():
            if (
                previous_vessel_statuses[vessel_id] == VesselStatus.APPROACHING
                and vessel.status == VesselStatus.WAITING
            ):
                operation_log.append(
                    {
                        "time": digital_twin.current_time,
                        "event": "vessel_arrival",
                        "vessel": vessel.name,
                        "details": f"Vessel {vessel.name} reached its ETA.",
                    }
                )

        for crane_id, crane in digital_twin.cranes.items():
            if (
                previous_crane_statuses[crane_id] != CraneStatus.BREAKDOWN
                and crane.status == CraneStatus.BREAKDOWN
            ):
                operation_log.append(
                    {
                        "time": digital_twin.current_time,
                        "event": "crane_breakdown",
                        "crane_id": crane.id,
                        "details": f"Crane {crane.id} entered breakdown status.",
                    }
                )

        if current_weather is not previous_weather:
            operation_log.append(
                {
                    "time": digital_twin.current_time,
                    "event": "weather_change",
                    "condition": current_weather.condition.value,
                    "details": f"Weather changed to {current_weather.condition.value}.",
                }
            )

        if step % 10 == 0:
            available_cranes = sum(
                crane.status == CraneStatus.AVAILABLE
                for crane in digital_twin.cranes.values()
            )
            berth_utilization = digital_twin.kpi_history["berth_utilization"][-1]
            print(
                f"Time: {digital_twin.current_time:.1f} hours; "
                f"weather: {current_weather.condition.value}; "
                f"available cranes: {available_cranes}; "
                f"berth-segment utilization: {berth_utilization:.1%}"
            )

    return operation_log


operation_log = simulate_digital_twin_operation(digital_twin, duration_hours=24)

print("\n=== Simulation summary ===")
print(f"Events logged: {len(operation_log)}")
print(f"Final simulation time: {digital_twin.current_time:.1f} hours")

if operation_log:
    print("\n=== First recorded events ===")
    for event in operation_log[:10]:
        print(f"{event['time']:.1f} hours: {event['details']}")
    if len(operation_log) > 10:
        print(f"Additional events not displayed: {len(operation_log) - 10}")
else:
    print("No arrival, weather-change, or crane-breakdown events were recorded.")

In [ ]:
def visualize_digital_twin_status(digital_twin: DigitalTwinSystem) -> None:
    """Plot simulated resource states, KPI histories, and weather records."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle("Terminal Simulation Status", fontsize=16, fontweight="bold")

    # Each rectangle represents one berth segment; color identifies occupancy.
    ax1 = axes[0, 0]
    ax1.set_title("Berth Segment Occupancy")
    ax1.set_xlabel("Quay position (m)")
    ax1.set_ylabel("Segment index")
    for i, segment in enumerate(digital_twin.berth_segments):
        occupied = not segment.is_available(digital_twin.current_time)
        color = "red" if occupied else "green"
        ax1.add_patch(
            Rectangle(
                (segment.start_position, i),
                segment.length,
                0.8,
                facecolor=color,
                edgecolor="black",
                alpha=0.7,
            )
        )
        if occupied and segment.occupied_by in digital_twin.vessels:
            ax1.text(
                segment.start_position + segment.length / 2,
                i + 0.4,
                f"V{segment.occupied_by}",
                ha="center",
                va="center",
            )
    ax1.set_xlim(0, digital_twin.quay_length)
    ax1.set_ylim(-1, max(1, len(digital_twin.berth_segments)))
    ax1.grid(True, alpha=0.3)
    ax1.legend(
        handles=[
            Rectangle((0, 0), 1, 1, color="green", label="Available"),
            Rectangle((0, 0), 1, 1, color="red", label="Occupied"),
        ],
        loc="upper right",
    )

    ax2 = axes[0, 1]
    ax2.set_title("Crane Status")
    ax2.set_xlabel("Crane ID")
    ax2.set_ylabel("Status indicator")
    status_colors = {
        CraneStatus.AVAILABLE: "green",
        CraneStatus.ASSIGNED: "blue",
        CraneStatus.MAINTENANCE: "orange",
        CraneStatus.BREAKDOWN: "red",
        CraneStatus.IDLE: "gray",
    }
    for crane_id, crane in digital_twin.cranes.items():
        ax2.bar(crane_id, 1, color=status_colors[crane.status], alpha=0.7)
        ax2.text(crane_id, 0.5, crane.status.value, ha="center", va="center", rotation=90)
    ax2.set_ylim(0, 1.2)
    ax2.set_xticks(list(digital_twin.cranes))
    ax2.grid(True, axis="y", alpha=0.3)

    ax3 = axes[0, 2]
    ax3.set_title("Vessel Status Counts")
    ax3.set_xlabel("Vessel status")
    ax3.set_ylabel("Number of vessels")
    status_counts: Dict[str, int] = {}
    for vessel in digital_twin.vessels.values():
        status_counts[vessel.status.value] = status_counts.get(vessel.status.value, 0) + 1
    if status_counts:
        ax3.bar(list(status_counts), list(status_counts.values()), color="skyblue", alpha=0.8)
    ax3.tick_params(axis="x", rotation=30)
    ax3.grid(True, axis="y", alpha=0.3)

    ax4 = axes[1, 0]
    ax4.set_title("Utilization Histories")
    ax4.set_xlabel("Simulation step")
    ax4.set_ylabel("Utilization (%)")
    if digital_twin.kpi_history["berth_utilization"]:
        time_steps = range(1, len(digital_twin.kpi_history["berth_utilization"]) + 1)
        berth_values = [100 * value for value in digital_twin.kpi_history["berth_utilization"]]
        crane_values = [100 * value for value in digital_twin.kpi_history["crane_utilization"]]
        ax4.plot(time_steps, berth_values, label="Berth-segment utilization")
        ax4.plot(time_steps, crane_values, label="Crane utilization")
        ax4.legend()
    ax4.set_ylim(0, 100)
    ax4.grid(True, alpha=0.3)

    ax5 = axes[1, 1]
    ax5.set_title("Recorded Weather Conditions")
    ax5.set_xlabel("Simulation time (hours)")
    ax5.set_ylabel("Condition")
    condition_order = [condition.value for condition in WeatherCondition]
    condition_map = {condition: index for index, condition in enumerate(condition_order)}
    weather_times = [record.timestamp for record in digital_twin.weather_history]
    weather_values = [condition_map[record.condition.value] for record in digital_twin.weather_history]
    ax5.plot(weather_times, weather_values, "o-", color="tab:green")
    ax5.set_yticks(range(len(condition_order)), condition_order)
    ax5.grid(True, alpha=0.3)

    ax6 = axes[1, 2]
    ax6.set_title("Current Simulation Indicators")
    ax6.axis("off")
    occupied_segments = sum(
        not segment.is_available(digital_twin.current_time)
        for segment in digital_twin.berth_segments
    )
    available_cranes = sum(
        crane.status == CraneStatus.AVAILABLE for crane in digital_twin.cranes.values()
    )
    latest_berth = (
        f"{digital_twin.kpi_history['berth_utilization'][-1]:.1%}"
        if digital_twin.kpi_history["berth_utilization"] else "Not yet recorded"
    )
    latest_crane = (
        f"{digital_twin.kpi_history['crane_utilization'][-1]:.1%}"
        if digital_twin.kpi_history["crane_utilization"] else "Not yet recorded"
    )
    current_weather = (
        digital_twin.weather_history[-1].condition.value
        if digital_twin.weather_history else "Not recorded"
    )
    metrics_text = (
        f"Simulation time: {digital_twin.current_time:.1f} hours\n"
        f"Vessel records: {len(digital_twin.vessels)}\n"
        f"Available cranes: {available_cranes}/{digital_twin.num_cranes}\n"
        f"Occupied berth segments: {occupied_segments}/{digital_twin.num_berth_segments}\n"
        f"Latest weather record: {current_weather}\n\n"
        f"Berth-segment utilization: {latest_berth}\n"
        f"Crane utilization: {latest_crane}\n"
        f"In-memory sensor records: {len(digital_twin.iot_sensors)}\n\n"
        "Allocation optimizer: not implemented\n"
        "Predictive models: not implemented\n"
        "External live data: not connected"
    )
    ax6.text(
        0.05,
        0.95,
        metrics_text,
        transform=ax6.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightblue", "alpha": 0.8},
    )

    plt.tight_layout()
    plt.show()


visualize_digital_twin_status(digital_twin)

## Interpreting the simulation

The simulation brings several terminal records into one evolving state, which is useful for understanding the data and synchronization requirements of a system-of-systems model. The event log and plots describe only the simulated transitions implemented above. For example, an arriving vessel can move from `APPROACHING` to `WAITING`, but it cannot progress to berth service unless assignment and service-time information are supplied by additional logic.

### Capabilities represented

- **Subsystem representation:** vessels, cranes, berth segments, weather records, and sensor records have explicit data structures.
- **Time-stepped state updates:** the system applies updates in a defined sequence at each simulation step.
- **Stochastic disturbances:** weather changes and crane breakdowns provide illustrative event sources.
- **State monitoring:** utilization histories and plots summarize selected simulated quantities.

### Capabilities outside this implementation

- **Live integration:** there are no AIS, IoT-device, weather-service, or terminal-operating-system connections.
- **Predictive analytics:** the prediction dictionaries are placeholders; no forecasting model is fitted or evaluated.
- **BAP-QCAP optimization:** no objective function, berth-allocation decisions, crane-allocation decisions, or allocation constraints are implemented here.
- **What-if analysis and rolling-horizon replanning:** no scenario generator or repeated optimization procedure is implemented.
- **Operational performance claims:** the simulation does not establish real-terminal utilization, turnaround-time improvements, prediction accuracy, or schedule adherence.

A production digital twin would require validated data sources, calibrated transition and disturbance models, operational decision logic, and suitable validation against observed terminal behavior. Those components are outside the scope of this notebook.

In [ ]:
def final_digital_twin_summary(
    digital_twin: DigitalTwinSystem,
    operation_log: List[Dict],
) -> None:
    """Report system configuration, simulated events, and available indicators."""
    print("=== TERMINAL SIMULATION SUMMARY ===")

    print("\nSYSTEM CONFIGURATION")
    print(f"Quay length: {digital_twin.quay_length} m")
    print(f"Berth segments: {digital_twin.num_berth_segments}")
    print(f"Quay cranes: {digital_twin.num_cranes}")
    print(f"In-memory sensor records: {len(digital_twin.iot_sensors)}")
    print(f"Planning horizon setting: {digital_twin.planning_horizon} hours")
    print(f"Simulation speed setting: {digital_twin.simulation_speed}x")

    print("\nSIMULATED STATE AND EVENTS")
    print(f"Vessel records: {len(digital_twin.vessels)}")
    print(f"Crane records: {len(digital_twin.cranes)}")
    print(f"Berth-segment records: {len(digital_twin.berth_segments)}")
    print(f"Weather records: {len(digital_twin.weather_history)}")
    print(f"Simulation time reached: {digital_twin.current_time:.1f} hours")
    print(f"Logged events: {len(operation_log)}")
    completed = sum(
        vessel.status == VesselStatus.COMPLETED
        for vessel in digital_twin.vessels.values()
    )
    print(f"Vessels in completed status: {completed}")
    print(f"Logged crane-breakdown events: {sum(e['event'] == 'crane_breakdown' for e in operation_log)}")
    print(f"Logged weather-change events: {sum(e['event'] == 'weather_change' for e in operation_log)}")

    print("\nRECORDED UTILIZATION INDICATORS")
    berth_history = digital_twin.kpi_history["berth_utilization"]
    crane_history = digital_twin.kpi_history["crane_utilization"]
    if berth_history and crane_history:
        print(f"Mean berth-segment utilization: {np.mean(berth_history):.1%}")
        print(f"Maximum berth-segment utilization: {max(berth_history):.1%}")
        print(f"Mean crane utilization: {np.mean(crane_history):.1%}")
        print(f"Maximum crane utilization: {max(crane_history):.1%}")
    else:
        print("Utilization indicators are unavailable because no time steps were simulated.")

    print("\nIMPLEMENTATION BOUNDARIES")
    print("Live external data integration: not implemented")
    print("Arrival and failure prediction: not implemented")
    print("Berth and crane allocation optimization: not implemented")
    print("Scenario analysis and rolling-horizon replanning: not implemented")
    print(
        "Interpret utilization as a property of the simulated assignments; "
        "this notebook does not create an allocation plan."
    )


final_digital_twin_summary(digital_twin, operation_log)